In [1]:
!pip install z3-solver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 41.7 MB/s  0:00:00


In [2]:
import time
import csv
import re
from datasets import load_dataset
import z3
import itertools
from collections import defaultdict

In [4]:
from huggingface_hub import notebook_login
notebook_login()

In [15]:
"""
General-purpose reasoning engine
"""
import itertools
import re
from collections import defaultdict

Z3_PROBLEM = None
try:
    import z3
    if not hasattr(z3, "DeclareSort"):
        try:
            from z3 import z3 as _real_z3
            if hasattr(_real_z3, "DeclareSort"):
                z3 = _real_z3
        except Exception:
            pass
    if not hasattr(z3, "DeclareSort"):
        Z3_PROBLEM = ("Imported module 'z3' (%s) is not the Z3 solver library. Run: "
                      "pip uninstall -y z3 ; pip install -U z3-solver ; then restart the runtime."
                      % getattr(z3, "__file__", "?"))
        z3 = None
except Exception as _e:
    z3 = None
    Z3_PROBLEM = "Could not import z3 (%s). Run: pip install -U z3-solver" % _e


#  FIRST-ORDER LOGIC  ->  Z3
_FOL_TOKEN = re.compile(r"<->|<=>|->|=>|!=|&&|\|\||[∀∃¬∧∨⊕→↔≠(),=&|~!^]|\w+")
_FOL_MAP = {
    "→": "->", "=>": "->", "↔": "<->", "<=>": "<->", "∧": "&", "&&": "&", "∨": "|", "||": "|",
    "¬": "~", "!": "~", "≠": "!=", "⊕": "^", "∀": "forall", "∃": "exists",
}
_WORD = re.compile(r"\w+$")


def _tokenize_fol(text):
    toks = []
    for t in _FOL_TOKEN.findall(str(text)):
        t = _FOL_MAP.get(t, t)
        if t.lower() in ("forall", "exists"):
            t = t.lower()
        toks.append(t)
    return toks


class _FOLParser:
    """precedence (low->high): <->  ->  (| ^)  &  ~ / quantifiers / atoms.  '->' is right assoc.
    A quantifier's body extends as far right as possible (standard convention)."""

    def __init__(self, text):
        self.t = _tokenize_fol(text)
        self.i = 0

    def peek(self, k=0):
        j = self.i + k
        return self.t[j] if j < len(self.t) else None

    def next(self):
        tok = self.peek()
        self.i += 1
        return tok

    def expect(self, tok):
        if self.next() != tok:
            raise ValueError("expected " + tok)

    def parse(self):
        f = self.iff()
        if self.peek() is not None:
            raise ValueError("trailing tokens")
        return f

    def iff(self):
        l = self.imp()
        while self.peek() == "<->":
            self.next()
            l = ("iff", l, self.imp())
        return l

    def imp(self):
        l = self.disj()
        if self.peek() == "->":
            self.next()
            return ("imp", l, self.imp())
        return l

    def disj(self):
        l = self.conj()
        while self.peek() in ("|", "^"):
            op = self.next()
            l = ("or" if op == "|" else "xor", l, self.conj())
        return l

    def conj(self):
        l = self.unary()
        while self.peek() == "&":
            self.next()
            l = ("and", l, self.unary())
        return l

    def unary(self):
        t = self.peek()
        if t is None:
            raise ValueError("unexpected end")
        if t == "~":
            self.next()
            return ("not", self.unary())
        if t in ("forall", "exists"):
            self.next()
            vs = []
            first = True
            while True:
                p = self.peek()
                if p == ",":
                    self.next()
                    continue
                if p is not None and _WORD.match(p) and p not in ("forall", "exists") \
                        and (first or self.peek(1) != "("):
                    vs.append(self.next())
                    first = False
                    continue
                break
            if not vs:
                raise ValueError("quantifier without variable")
            node = self.iff()
            for v in reversed(vs):
                node = (t, v, node)
            return node
        if t == "(":
            self.next()
            f = self.iff()
            self.expect(")")
            return f
        return self.atom()

    def term(self):
        t = self.next()
        if t is None or not _WORD.match(t) or t in ("forall", "exists"):
            raise ValueError("bad term " + str(t))
        if self.peek() == "(":
            self.next()
            args = []
            if self.peek() == ")":
                self.next()
                return ("app", t, (), True)
            while True:
                args.append(self.term())
                p = self.next()
                if p == ")":
                    break
                if p != ",":
                    raise ValueError("bad argument list")
            return ("app", t, tuple(args), True)
        return ("app", t, (), False)

    def atom(self):
        t = self.term()
        if self.peek() in ("=", "!="):
            op = self.next()
            n = ("eq", t, self.term())
            return ("not", n) if op == "!=" else n
        return ("atom", t)


class _Z3Builder:
    """Compiles FOL ASTs into Z3 over a single uninterpreted sort."""

    _FREE_VAR = re.compile(r"[uvwxyz]\d*$")

    def __init__(self):
        self.S = z3.DeclareSort("Obj")
        self.consts, self.funcs, self.preds, self.props = {}, {}, {}, {}
        self.n = 0
        self.free = {}

    def _const(self, name):
        if name not in self.consts:
            self.consts[name] = z3.Const("c_" + name, self.S)
        return self.consts[name]

    def term(self, t, env):
        _, name, args, paren = t
        if args:
            key = (name, len(args))
            if key not in self.funcs:
                self.funcs[key] = z3.Function("f_" + name, *([self.S] * len(args) + [self.S]))
            return self.funcs[key](*[self.term(a, env) for a in args])
        if name in env:
            return env[name]
        if not paren and self._FREE_VAR.match(name):
            if name not in self.free:
                self.free[name] = z3.Const("fv_" + name, self.S)
            return self.free[name]
        return self._const(name)

    def build(self, node, env):
        k = node[0]
        if k == "atom":
            _, name, args, _p = node[1]
            if not args:
                if name not in self.props:
                    self.props[name] = z3.Bool("p_" + name)
                return self.props[name]
            key = (name, len(args))
            if key not in self.preds:
                self.preds[key] = z3.Function("P_" + name, *([self.S] * len(args) + [z3.BoolSort()]))
            return self.preds[key](*[self.term(a, env) for a in args])
        if k == "eq":
            return self.term(node[1], env) == self.term(node[2], env)
        if k == "not":
            return z3.Not(self.build(node[1], env))
        if k == "and":
            return z3.And(self.build(node[1], env), self.build(node[2], env))
        if k == "or":
            return z3.Or(self.build(node[1], env), self.build(node[2], env))
        if k == "xor":
            return z3.Xor(self.build(node[1], env), self.build(node[2], env))
        if k == "imp":
            return z3.Implies(self.build(node[1], env), self.build(node[2], env))
        if k == "iff":
            return self.build(node[1], env) == self.build(node[2], env)
        if k in ("forall", "exists"):
            self.n += 1
            v = z3.Const("%s_%d" % (node[1], self.n), self.S)
            env2 = dict(env)
            env2[node[1]] = v
            body = self.build(node[2], env2)
            return z3.ForAll([v], body) if k == "forall" else z3.Exists([v], body)
        raise ValueError("unknown node " + str(k))

    def compile(self, text):
        self.free = {}
        f = self.build(_FOLParser(text).parse(), {})
        if self.free:
            f = z3.ForAll(list(self.free.values()), f)
        return f


#  ENGINE
_ART = {"the", "a", "an"}
_QVARS = {"something", "someone", "anything", "anyone", "everything", "everyone", "it", "they",
          "he", "she", "one"}
_GENERIC = {"things", "people", "thing", "person", "persons", "animals", "objects", "creatures",
            "ones", "individuals"}
_COPULAS = {"is", "are", "was", "were"}


def _lemma(w):
    w = w.lower()
    irregular = {"has": "have", "does": "do", "is": "be"}
    if w in irregular:
        return irregular[w]
    if w.endswith("ies") and len(w) > 4:
        return w[:-3] + "y"
    if re.search(r"(ch|sh|ss|x|z)es$", w):
        return w[:-2]
    if w.endswith("s") and not w.endswith("ss") and len(w) > 3:
        return w[:-1]
    return w


class GeneralPurposeReasoningEngine:
    def __init__(self, closed_world=False, z3_timeout_ms=3000):
        """
        closed_world : False -> a negative statement is true only if it is stated/derived (OWA);
                       True  -> negation as failure: 'not P' is true when P cannot be derived (CWA).
                       Rule-reasoning datasets differ here; try both on a validation split.
        """
        self.closed_world = closed_world
        self.z3_timeout_ms = z3_timeout_ms

    # routing
    @staticmethod
    def _looks_like_fol(text):
        s = str(text)
        return bool(re.search(r"[∀∃¬∧∨⊕→↔]", s)) or \
            bool(re.search(r"\b[A-Za-z_]\w*\s*\(\s*\w+(\s*,\s*\w+)*\s*\)", s))

    @staticmethod
    def _as_lines(x):
        if x is None:
            return []
        if isinstance(x, (list, tuple)):
            return [str(i).strip() for i in x if str(i).strip()]
        return [l.strip() for l in str(x).split("\n") if l.strip()]

    @staticmethod
    def _extract_fields(sample):
        lk = {str(k).lower().replace("-", "_").replace(" ", "_"): k for k in sample}

        def pick(cands):
            for c in cands:
                if c in lk and sample[lk[c]] not in (None, "", []):
                    return sample[lk[c]]
            return None
        return (pick(["premises_fol", "premisses_fol", "premises_folio", "fol_premises", "premise_fol"]),
                pick(["conclusion_fol", "conclusion_folio", "fol_conclusion", "hypothesis_fol"]),
                pick(["premises", "premisses", "premise", "context", "story", "theory", "given_info"]),
                pick(["conclusion", "question", "statement", "hypothesis", "query"]))

    def _detect_modality(self, context_input, query_input=None):
        if isinstance(context_input, dict):
            pf, cf, pn, cn = self._extract_fields(context_input)
            if pf is not None or (pn is not None and cn is not None):
                return "fol" if (pf is not None or self._looks_like_fol("\n".join(self._as_lines(pn)))) \
                    else "nl_pair"
            return "nl_pair"
        q = str(query_input or "")
        c = str(context_input or "")
        if re.match(r"\s*would\b", q, re.I) and re.search(r"\bcauses?\b", c, re.I):
            return "counterfactual"
        return "logical"

    def evaluate_sample(self, context, query=None):
        modality = self._detect_modality(context, query)
        if modality == "fol":
            return self._evaluate_fol_task(context)
        if modality == "nl_pair":
            if isinstance(context, dict):
                pf, cf, pn, cn = self._extract_fields(context)
                if pn is None or cn is None:
                    return "Uncertain"
                return self._three_way_nl(pn, cn)
        if modality == "counterfactual":
            return self._evaluate_counterfactual_task(context, query)
        return self._evaluate_logical_task(context, query)

    def evaluate_sample_3way(self, context, query=None):
        """Same routing, but always returns 'True' / 'False' / 'Uncertain'."""
        modality = self._detect_modality(context, query)
        if modality == "fol":
            return self._evaluate_fol_task(context)
        if modality == "nl_pair":
            pf, cf, pn, cn = self._extract_fields(context)
            return self._three_way_nl(pn, cn) if pn is not None and cn is not None else "Uncertain"
        if modality == "counterfactual":
            return "True" if self._evaluate_counterfactual_task(context, query) else "False"
        return self._three_way_nl(context, query)

    # MODALITY 1: FOL
    def _evaluate_fol_task(self, sample):
        if not sample or not isinstance(sample, dict):
            return "Uncertain"
        pf, cf, pn, cn = self._extract_fields(sample)
        prem, conc = None, None
        if pf is not None and cf is not None:
            prem, conc = self._as_lines(pf), str(cf).strip()
        elif pn is not None and cn is not None and self._looks_like_fol("\n".join(self._as_lines(pn))):
            prem, conc = self._as_lines(pn), str(cn).strip()
        if prem is not None:
            if z3 is None:
                raise RuntimeError("FOL sample needs the Z3 solver. " + str(Z3_PROBLEM))
            return self._fol_entails(prem, conc)
        if pn is not None and cn is not None:
            return self._three_way_nl(pn, cn)
        return "Uncertain"

    def _fol_entails(self, premises, conclusion):
        b = _Z3Builder()
        assertions = []
        for p in premises:
            try:
                assertions.append(b.compile(p))
            except Exception:
                continue
        try:
            c = b.compile(conclusion)
        except Exception:
            return "Uncertain"
        try:
            s = z3.Solver()
            s.set("timeout", int(self.z3_timeout_ms))
            for a in assertions:
                s.add(a)
            s.push()
            s.add(z3.Not(c))
            r1 = s.check()
            s.pop()
            if r1 == z3.unsat:
                return "True"
            s.push()
            s.add(c)
            r2 = s.check()
            s.pop()
            if r2 == z3.unsat:
                return "False"
        except Exception:
            pass
        return "Uncertain"

    # MODALITY 2: counterfactual SCM
    @staticmethod
    def _lit(text):
        m = re.fullmatch(r"\s*(not\s+)?(\w+)\s*", text, re.I)
        if not m:
            return None
        return m.group(2).lower(), not bool(m.group(1))

    def _parse_causal_story(self, text):
        """-> (mechanisms, observations).  mechanisms[effect] = [(negated_output, dnf)], where dnf is
        a list of conjunctions, each a list of (variable, required_value)."""
        text = str(text or "")
        obs = {}
        m = re.search(r"we\s+observed\b", text, re.I)
        if m:
            obs_part, text = text[m.end():], text[:m.start()]
            for piece in re.split(r"\band\b|,|;|\.", obs_part):
                lit = self._lit(piece)
                if lit:
                    obs[lit[0]] = lit[1]
        text = re.sub(r"^\s*we\s+know\s+that\s+", "", text, flags=re.I)
        mech = defaultdict(list)
        for clause in re.split(r",\s*(?:and\s+)?|\.\s*|;\s*", text):
            clause = clause.strip()
            if not clause:
                continue
            m = re.match(r"^(?:and\s+)?(.+?)\s+(?:together\s+)?causes?\s+(not\s+)?(\w+)$", clause, re.I)
            if not m:
                continue
            ante, neg_out, effect = m.group(1), bool(m.group(2)), m.group(3).lower()
            dnf = []
            for disj in re.split(r"\s+or\s+", ante, flags=re.I):
                conj = []
                for part in re.split(r"\s+and\s+", disj, flags=re.I):
                    lit = self._lit(part)
                    if lit:
                        conj.append(lit)
                if conj:
                    dnf.append(conj)
            if dnf:
                mech[effect].append((neg_out, dnf))
        return mech, obs

    @staticmethod
    def _scm_value_fn(mech, exo, do):
        memo = {}

        def val(v, stack=()):
            if v in do:
                return do[v]
            if v in memo:
                return memo[v]
            if v not in mech:
                return exo.get(v, False)
            if v in stack:
                return False
            out = False
            for neg_out, dnf in mech[v]:
                r = any(all(val(x, stack + (v,)) == pol for x, pol in conj) for conj in dnf)
                out = out or ((not r) if neg_out else r)
            memo[v] = out
            return out
        return val

    def _evaluate_counterfactual_task(self, story_text, query_text):
        mech, obs = self._parse_causal_story(story_text)
        q = str(query_text or "")
        m = re.search(r"would\s+(not\s+)?(\w+)\s+(?:occur\s+)?(?:if\s+(.+?))?"
                      r"(?:\s+instead\s+of\s+(.+?))?[\s\?\.]*$", q, re.I)
        if not m:
            return False
        neg_target, target = bool(m.group(1)), m.group(2).lower()
        do, factual = {}, {}
        if m.group(3):
            for part in re.split(r"\s+and\s+", m.group(3), flags=re.I):
                lit = self._lit(part)
                if lit:
                    do[lit[0]] = lit[1]
        if m.group(4):
            for part in re.split(r"\s+and\s+", m.group(4), flags=re.I):
                lit = self._lit(part)
                if lit:
                    factual[lit[0]] = lit[1]
        for v, pol in do.items():
            factual.setdefault(v, not pol)

        variables = set(mech) | {x for rs in mech.values() for _, dnf in rs for c in dnf for x, _ in c}
        variables |= set(obs) | set(do) | {target}
        exo = [v for v in variables if v not in mech]
        fixed = {v: obs[v] for v in exo if v in obs}
        for v in exo:
            if v in factual and v not in fixed:
                fixed[v] = factual[v]
        free = [v for v in exo if v not in fixed and v not in do]
        if len(free) > 14:
            candidates = [()]
            free_names = []
        else:
            candidates = itertools.product([False, True], repeat=len(free))
            free_names = free
        endo_obs = {v: b for v, b in obs.items() if v in mech}

        best = None
        for bits in candidates:
            assign = dict(fixed)
            assign.update(zip(free_names, bits))
            fac = self._scm_value_fn(mech, assign, {})
            if all(fac(v) == b for v, b in endo_obs.items()):
                cost = sum(bits)
                if best is None or cost < best[0]:
                    best = (cost, assign)
        assign = best[1] if best else dict(fixed)
        cf = self._scm_value_fn(mech, assign, do)
        result = cf(target)
        return (not result) if neg_target else result

    # MODALITY 3: rule reasoning
    @staticmethod
    def _norm(s):
        s = re.sub(r"\s+", " ", str(s)).strip().rstrip(".!?;").strip()
        s = re.sub(r"\b(is|are|does|do|was|were|has|have|can|will|would)n['’]t\b", r"\1 not", s, flags=re.I)
        return s

    @staticmethod
    def _term(toks):
        toks = [t for t in toks]
        while toks and toks[0] in _ART:
            toks = toks[1:]
        if not toks:
            return "unknown"
        if len(toks) == 1 and toks[0] in _QVARS:
            return "?x"
        return "_".join(toks)

    def _parse_atom(self, text):
        orig = self._norm(text).split()
        low = [w.lower() for w in orig]
        if not low:
            return None
        for idx, w in enumerate(low):
            if w in _COPULAS and idx >= 1:
                subj = self._term(low[:idx])
                rest, neg = low[idx + 1:], False
                if rest and rest[0] == "not":
                    neg, rest = True, rest[1:]
                while rest and rest[0] in _ART:
                    rest = rest[1:]
                return (neg, "_".join(rest), (subj,)) if rest else None
        for idx, w in enumerate(low):
            if w in ("does", "do") and idx >= 1 and idx + 2 < len(low) + 0 and low[idx + 1] == "not":
                subj = self._term(low[:idx])
                verb = _lemma(low[idx + 2]) if idx + 2 < len(low) else None
                if verb is None:
                    return None
                obj = low[idx + 3:]
                return (True, verb, (subj,) + ((self._term(obj),) if obj else ()))
        if len(low) < 2:
            return None
        j = None
        for i in range(2, len(low)):
            if low[i] in _ART or orig[i][:1].isupper():
                j = i
                break
        if j is not None:
            subj, verb, obj = low[:j - 1], low[j - 1], low[j:]
        elif len(low) == 2:
            subj, verb, obj = low[:1], low[1], []
        elif low[0] in _ART:
            subj, verb, obj = low[:-1], low[-1], []
        else:
            subj, verb, obj = low[:1], low[1], low[2:]
        args = (self._term(subj),) + ((self._term(obj),) if obj else ())
        return (False, _lemma(verb), args)

    def _atoms_from(self, text, strict=False):
        """Parse a conjunction into literals.  strict=True returns None if ANY part is unparsable
        (used for rules: a half-parsed body would make the rule fire far too often)."""
        lits = []
        for p in re.split(r"\s*,\s*(?:and\s+)?|\s+and\s+", str(text).strip()):
            if not p.strip():
                continue
            lit = self._parse_atom(p)
            if lit is None and lits:
                ws, neg = p.lower().split(), False
                if ws and ws[0] == "not":
                    neg, ws = True, ws[1:]
                while ws and ws[0] in _ART:
                    ws = ws[1:]
                if ws:
                    lit = (neg, "_".join(ws), lits[-1][2][:1])
            if lit:
                lits.append(lit)
            elif strict:
                return None
        if strict and not lits:
            return None
        return lits

    def _parse_sentence(self, sent):
        s = self._norm(sent)
        if not s:
            return []
        low = s.lower()
        items = []
        m = re.match(r"^(?:if|when|whenever|given that|assuming that|assuming)\s+(.+?)\s*(?:,\s*)?\bthen\b\s+(.+)$",
                     s, re.I) or \
            re.match(r"^(?:if|when|whenever|given that|assuming that|assuming)\s+([^,]+),\s*(.+)$", s, re.I)
        if m:
            body = self._atoms_from(m.group(1), strict=True)
            heads = self._atoms_from(m.group(2), strict=True)
            if body is None or heads is None:
                return []
            for h in heads:
                items.append(("rule", body, h))
            return items
        m = re.match(r"^(.+?)\s+if\s+(.+)$", s, re.I)
        if m:
            body = self._atoms_from(m.group(2), strict=True)
            heads = self._atoms_from(m.group(1), strict=True)
            if body is None or heads is None:
                return []
            for h in heads:
                items.append(("rule", body, h))
            return items
        owords = s.replace(",", " ").split()
        lwords = [w.lower() for w in owords]
        k = next((i for i, w in enumerate(lwords) if w in _GENERIC), None)
        if k is not None:
            pre = lwords[:k]
            if all(p not in _ART and p not in _COPULAS and p != "not" for p in pre):
                adjs = [p for p in pre if p not in {"all", "every", "each", "any", "and", "some"}]
                body = [(False, a, ("?x",)) for a in adjs]
                for h in self._atoms_from("something " + " ".join(owords[k + 1:])):
                    items.append(("rule", body, h))
                return items
        for lit in self._atoms_from(s):
            if any(a.startswith("?") for a in lit[2]):
                if re.match(r"(everything|everyone|all)\b", low):
                    items.append(("rule", [], lit))
                continue
            items.append(("fact", lit))
        return items

    @staticmethod
    def _sentences(context):
        if isinstance(context, (list, tuple)):
            raw = [str(c) for c in context]
        else:
            raw = str(context or "").split("\n")
        out = []
        for line in raw:
            line = line.strip()
            if not line or line.endswith(":"):
                continue
            out.extend(p.strip() for p in re.split(r"(?<=[.!?])\s+", line) if p.strip())
        return out

    @staticmethod
    def _stratify(rules):
        preds = {r[1][1] for r in rules} | {l[1] for r in rules for l in r[0]}
        s = {p: 0 for p in preds}
        for _ in range(len(preds) + 2):
            changed = False
            for body, head in rules:
                for neg, p, _a in body:
                    need = s[p] + (1 if neg else 0)
                    if s[head[1]] < need:
                        s[head[1]] = need
                        changed = True
            if not changed:
                return s
        return {}

    def _closure(self, facts, rules):
        pos, neg = defaultdict(set), defaultdict(set)
        for n, p, a in facts:
            (neg if n else pos)[p].add(a)
        consts = {a for _n, _p, args in facts for a in args if not a.startswith("?")}
        for body, head in rules:
            for _n, _p, args in list(body) + [head]:
                consts |= {a for a in args if not a.startswith("?")}
        consts = sorted(consts)
        cwa = self.closed_world
        strata = self._stratify(rules) if cwa else {}
        levels = sorted(set(strata.values())) or [0]

        def naf_holds(p, args):
            return args in neg[p] or (cwa and args not in pos[p])

        def match(pl, i, sigma):
            if i == len(pl):
                yield sigma
                return
            _n, p, pargs = pl[i]
            for fargs in list(pos[p]):
                if len(fargs) != len(pargs):
                    continue
                s2, ok = dict(sigma), True
                for pa, fa in zip(pargs, fargs):
                    if pa.startswith("?"):
                        if s2.setdefault(pa, fa) != fa:
                            ok = False
                            break
                    elif pa != fa:
                        ok = False
                        break
                if ok:
                    yield from match(pl, i + 1, s2)

        for lvl in levels:
            rs = [r for r in rules if strata.get(r[1][1], 0) == lvl]
            changed = True
            while changed:
                changed = False
                for body, head in rs:
                    pl = [l for l in body if not l[0]]
                    nl = [l for l in body if l[0]]
                    allv = {a for _n, _p, args in body + [head] for a in args if a.startswith("?")}
                    for sigma in match(pl, 0, {}):
                        free = [v for v in sorted(allv) if v not in sigma]
                        for combo in itertools.product(consts, repeat=len(free)):
                            s2 = dict(sigma)
                            s2.update(zip(free, combo))
                            if not all(naf_holds(p, tuple(s2.get(a, a) for a in args))
                                       for _n, p, args in nl):
                                continue
                            hn, hp, hargs = head
                            g = tuple(s2.get(a, a) for a in hargs)
                            tgt = neg if hn else pos
                            if g not in tgt[hp]:
                                tgt[hp].add(g)
                                changed = True
        return pos, neg

    def _solve_nl(self, context, query):
        facts, rules = [], []
        for sent in self._sentences(context):
            for it in self._parse_sentence(sent):
                if it[0] == "fact":
                    facts.append(it[1])
                else:
                    rules.append((it[1], it[2]))
        qlits = self._atoms_from(self._norm(query))
        if not qlits:
            return "Unknown"
        pos, neg = self._closure(facts, rules)
        verdicts = []
        for n, p, a in qlits:
            if not n:
                v = "True" if a in pos[p] else ("False" if a in neg[p] or self.closed_world else "Unknown")
            else:
                v = "True" if a in neg[p] else ("False" if a in pos[p] else
                                                 ("True" if self.closed_world else "Unknown"))
            verdicts.append(v)
        if all(v == "True" for v in verdicts):
            return "True"
        if any(v == "False" for v in verdicts):
            return "False"
        return "Unknown"

    # Legacy ground-fact pipeline
    @staticmethod
    def _legacy_predicate(text):
        text = text.lower().strip()
        m = re.match(r'^(?:the\s+)?([a-z0-9_]+)\s+is\s+(?:a\s+|an\s+)?([a-z0-9_]+)$', text)
        if m:
            subj, attr = m.groups()
            return (attr, (subj,))
        tokens = text.split()
        if len(tokens) == 2:
            return (tokens[1], (tokens[0],))
        if len(tokens) > 2:
            if tokens[0] in ("the", "a", "an"):
                subj, relation = tokens[1], "_".join(tokens[2:])
            else:
                subj, relation = tokens[0], "_".join(tokens[1:])
            return (relation, (subj,))
        return (text.replace(" ", "_"), ("unknown",))

    def _legacy_parse_sentence(self, sentence):
        sentence = str(sentence).strip().rstrip(".")
        if not sentence:
            return ("none", None)
        low = sentence.lower()
        if any(low.startswith(p) for p in ("if ", "when ", "given that ", "assuming ", "based on ")):
            cleaned = re.sub(r"^(if|when|given that|assuming|based on)\s+", "", low)
            parts = re.split(r",\s*(?:then|we can conclude|it follows that|we can say|it is evident that)\s+",
                             cleaned)
            if len(parts) == 2:
                ants = [a.strip() for a in re.split(r"\s+and\s+", parts[0].strip())]
                return ("rule", {"antecedents": [self._legacy_predicate(a) for a in ants],
                                 "consequent": self._legacy_predicate(parts[1].strip())})
        return ("fact", self._legacy_predicate(sentence))

    def _legacy_true(self, context, query):
        """True iff the legacy ground pipeline derives the query."""
        try:
            if isinstance(context, str):
                sents = []
                for line in [l.strip() for l in context.split("\n") if l.strip()]:
                    if line.endswith(":") or line.lower() in ("facts:", "rules:"):
                        continue
                    sents.extend(s.strip() for s in re.split(r"\.\s+", line) if s.strip())
            else:
                sents = list(context or [])
            facts, rules = [], []
            for s in sents:
                t, d = self._legacy_parse_sentence(s)
                if t == "fact" and d:
                    facts.append(d)
                elif t == "rule" and d:
                    rules.append(d)
            derived, added = set(facts), True
            while added:
                added = False
                for r in rules:
                    if r["consequent"] not in derived and all(a in derived for a in r["antecedents"]):
                        derived.add(r["consequent"])
                        added = True
            qt, qd = self._legacy_parse_sentence(query)
            return qt == "fact" and qd in derived
        except Exception:
            return False

    _LIFTED = re.compile(
        r"(?:^|[.!?]\s+|\b(?:if|when|whenever)\s+)(?:something|someone|somebody|anything|anyone|everything|"
        r"everyone|everybody|all|every|each|no one|nothing)\b|\b(?:things|people)\b", re.I)

    def _is_lifted(self, context):
        text = "\n".join(context) if isinstance(context, (list, tuple)) else str(context or "")
        return bool(self._LIFTED.search(text))

    def _three_way_nl(self, context, query):
        if self._legacy_true(context, query if not isinstance(query, (list, tuple)) else " ".join(query)):
            return "True"
        r = self._solve_nl(context, query if not isinstance(query, (list, tuple)) else " ".join(query))
        return "Uncertain" if r == "Unknown" else r

    def _evaluate_logical_task(self, context_input, query_sentence):
        if self._legacy_true(context_input, query_sentence):
            return True
        if self._is_lifted(context_input):          # quantified rules: need unification
            return self._solve_nl(context_input, query_sentence) == "True"
        return False

    def diagnose_logical(self, context, query):
        """Debug helper: show exactly how a context/query was parsed and what was derived."""
        facts, rules, skipped = [], [], []
        for sent in self._sentences(context):
            items = self._parse_sentence(sent)
            if not items:
                skipped.append(sent)
            for it in items:
                if it[0] == "fact":
                    facts.append(it[1])
                else:
                    rules.append((it[1], it[2]))
        pos, neg = self._closure(facts, rules)
        derived = sorted((p, a) for p, s in pos.items() for a in s if (False, p, a) not in facts)
        return {"facts": facts, "rules": rules, "unparsed_sentences": skipped,
                "query": self._atoms_from(self._norm(query)),
                "derived_positive": derived, "answer": self._solve_nl(context, query)}

In [16]:
if __name__ == "__main__":
    engine = GeneralPurposeReasoningEngine()
    submission_rows = []

    # 1. Evaluate on CounterBench Dataset
    print("Loading CounterBench dataset...")
    cb_dataset = load_dataset(
        "json",
        data_files="https://huggingface.co/datasets/CounterBench/CounterBench/raw/main/data_balanced_alpha_V1.json",
        split="train"
    )

    cb_correct = 0
    cb_total = len(cb_dataset)
    start_time = time.time()

    for idx, s in enumerate(cb_dataset):
        story = s.get('given_info', '')
        question = s.get('question', '')
        expected = s.get('answer', '').strip().lower()

        result_bool = engine.evaluate_sample(story, question)
        prediction = "yes" if result_bool else "no"

        if prediction == expected:
            cb_correct += 1

        submission_rows.append({"id": f"counterbench_{idx}", "predicted_label": result_bool})

    cb_inference_time = time.time() - start_time
    cb_accuracy = (cb_correct / cb_total) * 100 if cb_total > 0 else 0
    print(f"CounterBench Accuracy: {cb_accuracy:.2f}% ({cb_correct}/{cb_total}) | Time: {cb_inference_time:.2f}s")

    # 2. Evaluate on RuleTaker Dataset
    print("\nLoading RuleTaker dataset from Hugging Face...")
    rt_dataset = load_dataset("rikraj100/ruletaker", "default", split="test")

    rt_correct = 0
    rt_total = len(rt_dataset)
    start_time = time.time()

    for idx, sample in enumerate(rt_dataset):
        context = sample['context']
        question = sample['statement']
        label = True if str(sample['flag']).lower() == 'true' else False

        prediction = engine.evaluate_sample(context, question)
        if prediction == label:
            rt_correct += 1

        submission_rows.append({"id": f"ruletaker_{idx}", "predicted_label": str(prediction)})

    rt_inference_time = time.time() - start_time
    rt_accuracy = (rt_correct / rt_total) * 100 if rt_total > 0 else 0
    print(f"RuleTaker Accuracy: {rt_accuracy:.2f}% ({rt_correct}/{rt_total}) | Time: {rt_inference_time:.2f}s")

    # 3. Evaluate on FOLIO Dataset
    print("\nLoading FOLIO dataset from Hugging Face...")
    try:
        folio_dataset = load_dataset("yale-nlp/FOLIO", split="validation")
        folio_correct = 0
        valid_total = 0
        folio_total = len(folio_dataset)
        start_time = time.time()

        for idx, sample in enumerate(folio_dataset):
            label_val = sample.get('label')
            if label_val is None:
                continue

            expected = str(label_val).strip()
            prediction = engine.evaluate_sample(sample)

            if prediction == expected:
                folio_correct += 1
            valid_total += 1

            submission_rows.append({"id": f"folio_{idx}", "predicted_label": str(prediction)})

        folio_inference_time = time.time() - start_time
        folio_accuracy = (folio_correct / valid_total) * 100 if valid_total > 0 else 0
        print(f"FOLIO Accuracy: {folio_accuracy:.2f}% ({folio_correct}/{valid_total}) | Time: {folio_inference_time:.2f}s")
    except Exception as e:
        print(f"FOLIO skipped or error: {e}")

    # Generate submission.csv
    print("\nGenerating submission.csv...")
    with open("submission.csv", mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "predicted_label"])
        writer.writeheader()
        writer.writerows(submission_rows)
    print("submission.csv successfully created!")

Loading CounterBench dataset...
CounterBench Accuracy: 82.00% (820/1000) | Time: 0.23s

Loading RuleTaker dataset from Hugging Face...
RuleTaker Accuracy: 97.60% (244/250) | Time: 0.06s

Loading FOLIO dataset from Hugging Face...
FOLIO Accuracy: 67.49% (137/203) | Time: 1.53s

Generating submission.csv...
submission.csv successfully created!
